# Cassava Leaf Disease Classification — ML Pipeline Notebook

**African Leadership University — Machine Learning Pipeline Summative**

Run every cell top to bottom, in order, in a fresh Colab session with a **T4 GPU**
(`Runtime → Change runtime type → T4 GPU`). Don't skip cells or run out of order —
this notebook depends on state built up cell by cell.


## 0. Setup: clone repo + connect to GPU

In [ ]:
!git clone https://github.com/bianca255/cassava_ml_pipeline.git
%cd cassava_ml_pipeline/notebook


In [ ]:
!nvidia-smi


## 1. Data Acquisition\n\nSource: [Cassava Leaf Disease Classification (Kaggle Datasets mirror)](https://www.kaggle.com/datasets/nirmalsankalana/cassava-leaf-disease-classification) — a Kaggle **Dataset**, not the original Competition page, chosen because it doesn't require accepting competition rules or phone-number verification, and ships pre-organized into per-class folders.\n\n**Kaggle credentials**: Use Colab Secrets (key icon in the left sidebar) to store your Kaggle API token under a name like `colab-cassava`, then reference it below — never paste the token value directly into a cell.

In [ ]:
from google.colab import userdata
import os, json

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({
        "username": "biancahaguma",  # replace with your actual Kaggle username
        "key": userdata.get("colab-cassava")  # secret NAME, not the token value
    }, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)


In [ ]:
!mkdir -p ../data/raw
!kaggle datasets download -d nirmalsankalana/cassava-leaf-disease-classification -p ../data/raw
!unzip -o -q ../data/raw/cassava-leaf-disease-classification.zip -d ../data/raw
!ls ../data/raw/data


In [ ]:
import os, shutil

rename_map = {
    "Cassava___bacterial_blight": "Cassava Bacterial Blight (CBB)",
    "Cassava___brown_streak_disease": "Cassava Brown Streak Disease (CBSD)",
    "Cassava___green_mottle": "Cassava Green Mottle (CGM)",
    "Cassava___mosaic_disease": "Cassava Mosaic Disease (CMD)",
    "Cassava___healthy": "Healthy",
}
raw_data_dir = "../data/raw/data"
for old_name, new_name in rename_map.items():
    old_path = os.path.join(raw_data_dir, old_name)
    new_path = os.path.join(raw_data_dir, new_name)
    if os.path.exists(old_path):
        shutil.move(old_path, new_path)

print("Class folders:", os.listdir(raw_data_dir))


## 2. Data Preprocessing\n\nSplit into train/test, then build Keras generators with augmentation. **Critical**: images are preprocessed with EfficientNet's own `preprocess_input` function (imported inside `src/preprocessing.py`) rather than a manual `rescale=1./255` — EfficientNetB0 has its own built-in normalization, and pre-scaling on top of that corrupts the pretrained ImageNet features. This was a real bug found and fixed during development; see `src/preprocessing.py` for details.

In [ ]:
import sys
sys.path.append("..")

from src.preprocessing import split_dataset, get_data_generators, CLASS_NAMES, IMG_SIZE

split_dataset(raw_dir="../data/raw/data", out_dir="../data", test_size=0.2)


In [ ]:
import os
for split in ["train", "test"]:
    for cls in os.listdir(f"../data/{split}"):
        cls_path = f"../data/{split}/{cls}"
        if os.path.isdir(cls_path):
            n = len(os.listdir(cls_path))
            print(f"{split}/{cls}: {n} images")


In [ ]:
train_gen, val_gen, test_gen = get_data_generators(train_dir="../data/train", test_dir="../data/test")
print("Train batches:", len(train_gen), "| Val batches:", len(val_gen), "| Test batches:", len(test_gen))

# Sanity check: pixel range should be roughly 0-255 (raw), NOT 0.0-1.0.
# If this shows 0.0/1.0, the preprocessing fix did not take -- stop and check
# src/preprocessing.py before continuing.
images, labels = next(train_gen)
print("Batch shape:", images.shape)
print("Pixel min:", images.min(), "| Pixel max:", images.max())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    # de-normalize just for display purposes
    img = images[i] - images[i].min()
    img = img / (img.max() + 1e-8)
    ax.imshow(img)
    cls_idx = np.argmax(labels[i])
    ax.set_title(CLASS_NAMES[cls_idx], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


### Feature interpretation #1 — Class distribution

In [ ]:
from pathlib import Path
import seaborn as sns

counts = {cls: len(list(Path(f"../data/train/{cls}").glob("*.*"))) for cls in CLASS_NAMES}
plt.figure(figsize=(8,4))
sns.barplot(x=list(counts.keys()), y=list(counts.values()))
plt.xticks(rotation=30, ha="right")
plt.ylabel("Image count")
plt.title("Training set class distribution")
plt.tight_layout()
plt.show()
print(counts)


**Interpretation:** Cassava Mosaic Disease (CMD) dominates the training set, reflecting its real-world prevalence as the most common cassava disease in East African fields. This imbalance is why the pipeline uses class-weighted loss (below) rather than relying on raw accuracy alone.

### Feature interpretation #2 — Average brightness per class

In [ ]:
from PIL import Image

brightness = {}
for cls in CLASS_NAMES:
    files = list(Path(f"../data/train/{cls}").glob("*.*"))[:40]
    vals = [np.array(Image.open(f).convert("L")).mean() for f in files]
    if vals:
        brightness[cls] = np.mean(vals)

plt.figure(figsize=(8,4))
sns.barplot(x=list(brightness.keys()), y=list(brightness.values()))
plt.xticks(rotation=30, ha="right")
plt.ylabel("Average brightness (0-255)")
plt.title("Average image brightness per class")
plt.tight_layout()
plt.show()
print(brightness)


**Interpretation:** Diseases causing visible discoloration or necrosis (e.g. CBSD, CBB) shift average brightness relative to Healthy leaves, which tend to be more uniformly green. This gives the model a genuine low-level visual signal beyond texture alone.

### Feature interpretation #3 — Image resolution consistency

In [ ]:
res = {}
for cls in CLASS_NAMES:
    files = list(Path(f"../data/train/{cls}").glob("*.*"))[:40]
    dims = [Image.open(f).size for f in files]
    if dims:
        res[cls] = (np.mean([d[0] for d in dims]), np.mean([d[1] for d in dims]))

for cls, (w, h) in res.items():
    print(f"{cls:40s} avg_width={w:.0f}  avg_height={h:.0f}")


**Interpretation:** Consistent resolution across classes indicates images were captured under similar field-survey conditions, reducing the chance the model learns capture-device artifacts rather than genuine disease features.

## 3. Model Creation\n\nEfficientNetB0 pretrained on ImageNet, partially fine-tuned (layers 200+ trainable), with a dense head using L2 regularization, batch normalization, and dropout (0.4).

In [ ]:
from src.model import build_model, train_model, evaluate_model, save_model_as_h5, MODEL_PATH

model = build_model()
model.summary()


### Class weights\n\nCompensates for the class imbalance shown above.

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_labels = train_gen.classes
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(class_labels),
    y=class_labels
)
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)


## 4. Model Training\n\nEarly stopping, LR scheduling on plateau, and checkpointing the best epoch. **Watch the first 3-4 epochs**: `val_accuracy` should vary between epochs (a sign the model is learning) rather than freezing at one repeated value (a sign of a collapsed, degenerate model that always predicts one class).

In [ ]:
import tensorflow as tf

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7),
        tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True),
    ],
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout()
plt.show()


## 5. Model Testing / Evaluation\n\nFour+ metrics: **accuracy, precision (macro), recall (macro), F1 (macro)**, plus loss, confusion matrix, and full classification report — evaluated on the held-out test set **that the model never saw during training**.

In [ ]:
save_model_as_h5(model)
print(f"Model saved to: {MODEL_PATH}")

metrics = evaluate_model(model, test_gen)
print(f"Loss:      {metrics['loss']:.4f}")
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision_macro']:.4f}")
print(f"Recall:    {metrics['recall_macro']:.4f}")
print(f"F1 score:  {metrics['f1_macro']:.4f}")


In [ ]:
cm = np.array(metrics["confusion_matrix"])
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()


In [ ]:
import json
print(json.dumps(metrics["classification_report"], indent=2))


## 6. Single-Image Prediction Demo\n\nMirrors what the `/predict` API endpoint does.

In [ ]:
from src.prediction import predict_image

sample_file = next(Path("../data/test").rglob("*.jpg"), None)
if sample_file:
    result = predict_image(str(sample_file))
    print("True folder label:", sample_file.parent.name)
    print("Prediction:", result["predicted_class"], f"({result['confidence']*100:.1f}% confidence)")
    plt.imshow(plt.imread(sample_file))
    plt.title(f"Predicted: {result['predicted_class']}")
    plt.axis("off")
    plt.show()


## 7. Retraining Demonstration

Simulates the pipeline's retraining trigger: new bulk-uploaded images are ingested into
`data/train/<class>/`, and the model is fine-tuned further from its existing saved weights
(not from scratch), then re-evaluated. This mirrors exactly what `POST /retrain` on the
FastAPI backend does (`src/model.py::retrain()`).


In [ ]:
from src.model import retrain

new_metrics = retrain(epochs=3)
print(f"Retrained accuracy: {new_metrics['accuracy']:.4f}")
print(f"Retrained F1:       {new_metrics['f1_macro']:.4f}")


In [ ]:
from google.colab import files
files.download(MODEL_PATH)


## Summary\n\n- Data acquired from Kaggle's Cassava Leaf Disease Classification dataset mirror, reorganized into per-class folders, split 80/20 train/test.\n- Preprocessing uses EfficientNet's own `preprocess_input` (not a manual rescale), plus augmentation; three feature-level visual interpretations analyzed above.\n- Model: EfficientNetB0 transfer learning, partially fine-tuned, regularized (L2 + dropout + batch norm), trained with class weights, early stopping, and LR scheduling.\n- Evaluated with accuracy, macro precision/recall/F1, confusion matrix, classification report on a held-out test set.\n- Retraining demonstrated as an incremental fine-tune from the saved model on newly ingested data, matching the `/retrain` API endpoint used in production.